# 02 Explore weather (Open-Meteo) and fires (NASA FIRMS)

## 1. Weather needs no key. Ask for the last week and look at the shape

In [1]:
import io
import os

import pandas as pd
import polars as pl
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

weather = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 28.6139,
        "longitude": 77.2090,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,"
        "wind_direction_10m,precipitation,boundary_layer_height",
        "past_days": 7,
        "forecast_days": 1,
        "timezone": "GMT",
    },
    timeout=30,
).json()
print(weather.keys())
print(weather["hourly_units"])
weather_pd = pd.DataFrame(weather["hourly"])
weather_pd.tail(8)

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
{'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'wind_speed_10m': 'km/h', 'wind_direction_10m': '°', 'precipitation': 'mm', 'boundary_layer_height': 'm'}


,time,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation,boundary_layer_height
184,2026-11-12T16:00,22.5,74,2.7,203,0.0,15.0
185,2026-11-12T17:00,21.7,77,4.2,205,0.0,25.0
186,2026-11-12T18:00,21.4,76,4.9,204,0.0,30.0
187,2026-11-12T19:00,20.9,75,4.5,217,0.0,30.0
188,2026-11-12T20:00,20.5,76,1.5,249,0.0,15.0
189,2026-11-12T21:00,19.8,80,1.4,320,0.0,15.0
190,2026-11-12T22:00,19.3,82,1.9,287,0.0,20.0
191,2026-11-12T23:00,19.1,82,2.0,260,0.0,20.0


## 2. What to check in the weather

In [2]:
print(weather_pd.isna().sum())     # is any variable empty?
print(weather_pd["time"].iloc[0])   # no timezone in the text: it is UTC only because we asked
print(weather_pd["time"].iloc[-1])  # later than now? then these are forecast hours

time                     0
temperature_2m           0
relative_humidity_2m     0
wind_speed_10m           0
wind_direction_10m       0
precipitation            0
boundary_layer_height    0
dtype: int64
2026-11-05T00:00
2026-11-12T23:00


## 3. Fire detections. Never print the URL: your key is inside it
Box = Punjab + Haryana (west,south,east,north). Last 5 days.

In [3]:
key = os.environ["FIRMS_MAP_KEY"]
url = (
    "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{key}/VIIRS_NOAA20_NRT/73.8,27.6,77.7,32.6/5"
)
text = requests.get(url, timeout=60).text
print(text[:300])

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
29.43811,76.87774,330.03,0.79,0.78,2026-11-08,658,N20,VIIRS,n,2.0NRT,292.3,3.6,D
30.32354,76.66158,326.97,0.77,0.77,2026-11-08,658,N20,VIIRS,n,2.0NRT,294.48,5.16,D
30.61091,76.4


## 4. Read everything as text first, so nothing is silently converted

In [4]:
fires_pd = pd.read_csv(io.StringIO(text), dtype=str)
fires_pl = pl.read_csv(io.StringIO(text), infer_schema_length=0)

print(fires_pd["confidence"].value_counts())
print(fires_pd["acq_time"].str.len().value_counts())  # 3-digit times have lost a leading zero
fires_pl.group_by("acq_date").len().sort("acq_date")

confidence
n    1546
l      66
Name: count, dtype: int64
acq_time
3    1528
4      84
Name: count, dtype: int64


acq_date,len
str,u64
"""2026-11-08""",539
"""2026-11-09""",425
"""2026-11-10""",463
"""2026-11-11""",185


## Findings → decisions (the real product of step 3)
Confirm each row against your own output; add rows for anything new. Full detail: `.field-guide/DATASETS.md`.

| What we saw | Decision | Where it ends up |
|---|---|---|
| Locations mix reference monitors and low-cost sensors | Keep only `isMonitor` locations | extract/openaq.py |
| Some stations silent for weeks (caaqm, StateAir: years) | Skip stations silent > 7 days | config.py |
| One location holds several pm25 sensors; old ones dead since 2018 | Pick the pm25 sensor with the latest `datetimeLast` | extract/openaq.py |
| Hourly buckets start at :30 UTC (= on the hour IST); each has `coverage.percentComplete` | Keep `datetimeFrom.utc` as the hour key; keep coverage | transform/ |
| 234 of 264 hours received at R K Puram (11% missing) | Do not fill gaps; city mean needs 5+ stations | marts |
| Nulls, negative error codes, values above instrument range | Drop outside valid range per pollutant | config.py, transform/ |
| Same hour can arrive twice | Keep last copy per sensor+hour | transform/, DB primary key |
| Timestamps are UTC; Delhi is +05:30 | Store UTC; convert to IST only in models | stg_air_quality.sql |
| Max 1,000 rows/request; 60 req/min | Page through results, sleep 1.1 s between calls | extract/openaq.py |
| Weather times have no tz and run into the forecast | Ask for GMT; drop hours later than now | extract/weather.py, transform/ |
| FIRMS time is HHMM number losing leading zeros | Pad to 4 digits, build one UTC timestamp | transform/ |
| FIRMS confidence is a letter: `l`, `n`, `h` (saw 1546 n, 66 l) | Keep `n` and `h` | transform/ |
| FIRMS "last 5 days" returned only 4 dates | Ask for a window wider than needed; dedupe | extract/firms.py |
| Worst hours flatten at the same maximum | Cannot fix; record as a limit | docs/project_brief.md, README.md |
